# 003 · String Formatting

> **Syntax › Data Types** · Python 3.14

Python's **f-strings** are the direct counterpart of C#'s `$"..."` interpolated strings — and the format-spec mini-language after the `:` plays the role of .NET format strings like `F2` or `N0`. Python 3.14 adds **t-strings**, a template variant for safe, deferred processing.

**In this notebook you will learn:**

- f-strings and how they map to C# string interpolation
- the format specification mini-language: precision, width, alignment, separators, percent, hex
- the `{x=}` self-documenting specifier and `!r` / `!s` conversions
- the legacy `str.format()` and `%` styles (so you can read older code)
- formatting dates inside f-strings
- template strings (t-strings, PEP 750) and what they are for

## C# mental model

| C# | Python |
|---|---|
| `$"Hello {name}"` | `f"Hello {name}"` |
| `$"{price:F2}"` | `f"{price:.2f}"` |
| `$"{name,10}"` / `$"{name,-10}"` | `f"{name:>10}"` / `f"{name:<10}"` |
| `string.Format("{0} {1}", a, b)` | `"{} {}".format(a, b)` |
| `FormattableString` | `string.templatelib.Template` (t-string) |
| `$"{{literal}}"` | `f"{{literal}}"` |

The prefix letter is `f` (or `F`) instead of `$`, and alignment/width go *inside* the format spec rather than after a comma.

## f-strings basics

Any expression can go inside the braces — attribute access, method calls, arithmetic, even conditional expressions.

```csharp
// C#
var name = "alice";
var qty = 3;
Console.WriteLine($"{name} ordered {qty} item{(qty == 1 ? "" : "s")}");
```

In [ ]:
name = "alice"
qty = 3
price = 9.5

print(f"{name} ordered {qty} item{'' if qty == 1 else 's'}")
print(f"Total: {qty * price}")
print(f"Upper: {name.upper()}, length: {len(name)}")
print(f"Braces: {{literal}} and {name}")    # double braces escape, same as C#

## The format specification mini-language

Everything after the `:` inside the braces is a **format spec**: `[[fill]align][sign][#][0][width][,|_][.precision][type]`. The common cases:

| Goal | C# | Python | Result for `1234.5678` |
|---|---|---|---|
| 2 decimals | `{x:F2}` | `{x:.2f}` | `1234.57` |
| Thousands separator | `{x:N2}` | `{x:,.2f}` | `1,234.57` |
| Scientific | `{x:E2}` | `{x:.2e}` | `1.23e+03` |
| Percent | `{r:P1}` | `{r:.1%}` | (for `0.256`) `25.6%` |
| Hex | `{n:X}` | `{n:X}` / `{n:x}` | (for `255`) `FF` / `ff` |
| Right-align width 10 | `{x,10}` | `{x:>10}` | |
| Zero-padded | `{x:0000.000}` | `{x:08.3f}` | `1234.568` |

In [ ]:
x = 1234.5678

print(f"{x:.2f}")       # 1234.57    — fixed-point, 2 decimals
print(f"{x:,.2f}")      # 1,234.57   — with thousands separator
print(f"{x:_.2f}")      # 1_234.57   — underscore separator
print(f"{x:.2e}")       # 1.23e+03   — scientific
print(f"{x:08.1f}")     # 001234.6   — zero-padded to width 8
print(f"{3.14159:08.3f}")   # 0003.142

In [ ]:
n = 255
ratio = 0.256

print(f"{n:x} {n:X} {n:#x} {n:#06x}")   # ff FF 0xff 0x00ff
print(f"{n:b} {n:o}")                   # 11111111 377   — binary, octal
print(f"{ratio:%}")                     # 25.600000%  — multiplies by 100
print(f"{ratio:.1%}")                   # 25.6%
print(f"{1_000_000:,}")                 # 1,000,000
print(f"{42:+d} {-42:+d}")              # +42 -42  — always show sign

### Width, alignment, and fill

`<` left, `>` right, `^` centre, optionally preceded by a fill character. Numbers right-align by default, strings left-align (C# right-aligns both for positive widths).

In [ ]:
for item, cost in [("apple", 1.5), ("watermelon", 12.25), ("fig", 0.4)]:
    print(f"|{item:<12}|{cost:>8.2f}|")

print(f"[{'hi':^10}]")      # [    hi    ]
print(f"[{'hi':*^10}]")     # [****hi****]
print(f"[{'title':-^20}]")  # [-------title--------]

Width and precision can themselves be expressions — nest braces inside the spec:

In [ ]:
width, digits = 12, 3
print(f"[{x:{width}.{digits}f}]")    # [    1234.568]

## Self-documenting expressions: `{x=}`

Adding `=` prints the expression text **and** its value — great for quick debugging. C# has no equivalent (the closest is `nameof(x)` plus the value). You can still add a format spec after the `=`.

In [ ]:
user_id = 42
balance = 1234.5678

print(f"{user_id=}")              # user_id=42
print(f"{balance=:.2f}")          # balance=1234.57
print(f"{user_id * 2 = }")        # spaces around = are preserved
print(f"{name.upper()=}")

## Conversions: `!r`, `!s`, `!a`

- `!s` calls `str()` (the default — like `ToString()`)
- `!r` calls `repr()` — the developer-oriented representation, which puts quotes around strings. This is very useful in logs to spot stray whitespace.
- `!a` calls `ascii()` — like `repr` but escapes non-ASCII characters

In [ ]:
value = "  padded "
print(f"value is {value}")      # value is   padded
print(f"value is {value!r}")    # value is '  padded '  — whitespace visible
print(f"{'café'!a}")            # 'caf\xe9'

Note that `{x=}` uses `repr` by default — that's why `{name.upper()=}` printed quotes above.

## Formatting dates

Objects can define their own format spec. For `datetime` the spec is a `strftime` pattern — similar to C#'s `{date:yyyy-MM-dd}`, but with `%` codes.

| C# | Python | Meaning |
|---|---|---|
| `yyyy` | `%Y` | 4-digit year |
| `MM` | `%m` | month 01–12 |
| `dd` | `%d` | day 01–31 |
| `HH` / `mm` / `ss` | `%H` / `%M` / `%S` | 24-h time |
| `dddd` | `%A` | weekday name |
| `MMMM` | `%B` | month name |

In [ ]:
from datetime import datetime

d = datetime(2025, 3, 7, 14, 5, 9)
print(f"{d:%Y-%m-%d %H:%M}")     # 2025-03-07 14:05  (C#: {d:yyyy-MM-dd HH:mm})
print(f"{d:%A, %d %B %Y}")       # Friday, 07 March 2025
print(f"{d:%H:%M:%S}")
print(f"{d.isoformat()}")        # ISO 8601 — C#: d.ToString("o")

## Multi-line f-strings and nested quotes

Since **Python 3.12** (PEP 701), f-strings are parsed by the real grammar. That means:

- you can reuse the **same quote type** inside the braces (before 3.12 you had to alternate `'` and `"`)
- expressions inside braces can span multiple lines and even contain `#` comments
- triple-quoted f-strings work as you'd expect for multi-line output

In [ ]:
order = {"id": 1001, "customer": "bob", "items": ["pen", "notebook"]}

print(f"Order {order["id"]} for {order["customer"]}")    # same quotes — OK since 3.12
print(f"Items: {", ".join(order["items"])}")

report = f"""
Order:    {order["id"]}
Customer: {order["customer"].title()}
Count:    {len(order["items"])}
"""
print(report)

## Legacy styles: `str.format()` and `%`

You will see these in older code and some libraries. Prefer f-strings in new code.

`str.format()` is the closest thing to `string.Format`. It supports positional and named placeholders and the same format spec:

In [ ]:
print("{} bought {} items".format("alice", 3))
print("{0} and {1}, then {0} again".format("x", "y"))    # like string.Format("{0} {1} {0}")
print("{name}: {total:,.2f}".format(name="bob", total=1234.5))

template = "{product:<10}{price:>8.2f}"      # a reusable format template
print(template.format(product="widget", price=4.5))

`%`-formatting is the old C `printf` style. You'll still see it in the `logging` module, where it is the recommended form because formatting is deferred until a message is actually emitted.

In [ ]:
print("%s is %d years old" % ("alice", 30))
print("%.2f | %5s | %-5s|" % (3.14159, "ab", "cd"))
print("%(name)s has %(count)d" % {"name": "bob", "count": 2})

The built-in `format(value, spec)` applies a spec without an f-string — like `value.ToString("N2")`:

In [ ]:
print(format(1234.5678, ",.2f"))   # C#: 1234.5678.ToString("N2")
print(format(5, "08b"))            # 00000101 — C#: Convert.ToString(5, 2).PadLeft(8, '0')

## Template strings (t-strings) — new in Python 3.14

PEP 750 introduces the `t` prefix. A t-string looks like an f-string but **does not produce a `str`**. It produces a `string.templatelib.Template` object that keeps the literal parts and the interpolated values *separate*, so a library can process them safely before combining them — escaping HTML, parameterising SQL, structured logging, and so on.

This is the same idea as C#'s `FormattableString` (and interpolated string handlers): when a `$"..."` string is assigned to `FormattableString`, the format and arguments stay separate — which is how EF Core's `FromSqlInterpolated` turns interpolations into SQL parameters.

```csharp
// C#
FormattableString query = $"SELECT * FROM users WHERE name = {name}";
Console.WriteLine(query.Format);            // SELECT * FROM users WHERE name = {0}
Console.WriteLine(query.GetArgument(0));    // the value, kept separate
```

In [ ]:
from string.templatelib import Template, Interpolation

name = "bob"
tmpl = t"Hello {name}!"

print(type(tmpl))               # <class 'string.templatelib.Template'> — not a str
print(tmpl.strings)             # ('Hello ', '!')
print(tmpl.values)              # ('bob',)
print(tmpl.interpolations[0])   # Interpolation(value, expression, conversion, format_spec)

Iterating a `Template` yields the static strings and `Interpolation` objects in order. A processing function decides what to do with each value. Here is a tiny HTML-escaping renderer:

In [ ]:
import html
from string.templatelib import Template, Interpolation

def render_html(template: Template) -> str:
    parts = []
    for part in template:
        if isinstance(part, Interpolation):
            parts.append(html.escape(str(part.value)))   # escape only the *values*
        else:
            parts.append(part)                           # trusted literal text
    return "".join(parts)

comment = "<script>alert('hi')</script>"
print(render_html(t"<p>{comment}</p>"))
print(f"<p>{comment}</p>")       # an f-string can't tell literal text from data

⚠️ **Gotcha:** a t-string is **not** a string — there is no automatic conversion to `str`, and `print(tmpl)` shows the `Template` repr rather than rendered text. You always need a processing function (your own or a library's).

In [ ]:
try:
    tmpl + " and more"
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## Summary

| Task | C# | Python |
|---|---|---|
| Interpolation | `$"Hi {name}"` | `f"Hi {name}"` |
| 2 decimals | `{x:F2}` | `{x:.2f}` |
| Thousands | `{x:N0}` | `{x:,.0f}` (or `{n:,}` for ints) |
| Percent | `{r:P1}` | `{r:.1%}` |
| Hex | `{n:X}` | `{n:X}` |
| Right / left align | `{s,10}` / `{s,-10}` | `{s:>10}` / `{s:<10}` |
| Centre with fill | — | `{s:*^10}` |
| Debug print | `$"{nameof(x)}={x}"` | `f"{x=}"` |
| Quoted repr | — | `f"{s!r}"` |
| Date | `{d:yyyy-MM-dd}` | `{d:%Y-%m-%d}` |
| Composite format | `string.Format("{0}", a)` | `"{0}".format(a)` |
| Deferred / safe template | `FormattableString` | `t"..."` → `Template` |

## Exercises

1. Given `products = [("pen", 1.2, 10), ("notebook", 3.75, 4), ("stapler", 12.0, 1)]` (name, unit price, quantity), print an aligned table with columns *Product* (left, width 10), *Qty* (right, width 4), and *Total* (right, width 9, 2 decimals, thousands separator).
2. Print `0.07125` as a percentage with 1 decimal place, and `48879` as uppercase hex with a `0x` prefix.
3. Write a t-string processor `shout(template)` that upper-cases every interpolated value but leaves the literal text unchanged. `shout(t"hi {name}, you are {age}")` with `name="bob", age=30` → `"hi BOB, you are 30"`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
products = [("pen", 1.2, 10), ("notebook", 3.75, 4), ("stapler", 12.0, 1)]
print(f"{'Product':<10}{'Qty':>4}{'Total':>9}")
for name, unit, qty in products:
    print(f"{name:<10}{qty:>4}{unit * qty:>9,.2f}")

# 2
print(f"{0.07125:.1%}")      # 7.1%
print(f"{48879:#X}")         # 0XBEEF  — note '#X' upper-cases the prefix too
print(f"0x{48879:X}")        # 0xBEEF

# 3
from string.templatelib import Template, Interpolation

def shout(template: Template) -> str:
    return "".join(
        str(part.value).upper() if isinstance(part, Interpolation) else part
        for part in template
    )

name, age = "bob", 30
print(shout(t"hi {name}, you are {age}"))   # hi BOB, you are 30
```

</details>